# 06m — Generalization: Breakfast (action-segmentation family, Phase 2) — **PRIMARY**

PAPER_TODO §2.1. Breakfast (|V|~=48, N=1712, **10 activity classes**) is the closest external analog to
Smartflat (cooking actions, temporal phases) and the **only** action-seg dataset with enough videos per
class for the order-null. It was pre-registered as the **counterpoint to SDS2's 0/15**: *does symbol
order discriminate activities beyond frequency, on data with genuine recipe-phase ordering?*

**Outcome (see §3 and the closing Result cell): the question could not be asked here.** The top-6 activities have near-disjoint
action vocabularies, so the frequency histogram alone separates them perfectly (`auc_intact = 1.0` on
all 60 cells) and the order-null has **no headroom**. The 0/60 is genuine but reflects **task
degeneracy**, not evidence about ordering. A frequency-controlled task is needed for that.

**Bounds (reported honestly):** 10 classes -> C(10,2)=45 pairs x 4 grid combos is intractable, so the
order-null is run on the **top-6 activities by video count** (C(6,2)=15 comparisons) at `n_shuffles=200`
(the `order_information` library default — note this is **not** SDS2's protocol, which ran
`n_shuffles=100` with 2x5 CV; RH §15). The quality half covers **all 10** activities on a **<=50/class subsample**
to bound the O(n^2) pairwise-rTWE that `k_medoid` triggers. Kernel: `smartflat_repro`.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault('NUMBA_THREADING_LAYER', 'workqueue')  # fork-safe rTWE under nbconvert
os.environ.setdefault('MPLBACKEND', 'agg')                   # headless figures
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils import upsample_sequence
from smartflat.features.symbolic_barycenter.generalization.action_segmentation import (
    download_action_seg, load_action_seg, build_action_seg_ground_cost,
    dedup_by_execution, default_root, read_mapping, DATASETS)
from smartflat.features.symbolic_barycenter.generalization.suite import run_generalization_suite
from smartflat.features.symbolic_barycenter.registries import default_baseline_methods
from smartflat.features.symbolic_barycenter.visualization import plot_cohort_barycenters

NAME, UPSAMPLE = 'breakfast', 24   # UPSAMPLE ~= 2x median segment count (measured: median 6, max 25)
OUT = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'generalization', NAME)
os.makedirs(OUT, exist_ok=True)
print('dataset:', NAME, '| upsample_to:', UPSAMPLE, '| output dir:', OUT)

dataset: breakfast | upsample_to: 24 | output dir: /home/perochon/data-gold-final/outputs/symbolic_barycenter/generalization/breakfast


## 1. Load + config-vs-real-files sanity

In [2]:
# GT action labels used DIRECTLY as symbols (RLE'd -> segment sequences, background -> 0).
# download_action_seg pulls only the tiny GT text via HTTP Range (never the 30 GB features);
# it is flag-guarded, so this is a no-op once the data is cached.
download_action_seg(NAME)
meta, X, labels, G = load_action_seg(NAME)

# LEAKAGE GUARD (audit E4 / RESULTS_HANDOFF §27). Breakfast films each execution from up to 5
# cameras whose RLE'd GT is near-identical (98.8% byte-identical after RLE), so its 1712 "videos"
# are only ~503 independent executions. The order-null's RepeatedStratifiedKFold is NOT group-aware,
# so leaving duplicate views in puts copies of one execution in train AND test -> the classifier
# recognises rather than generalises. dedup_by_execution keeps one view per (subject, trial,
# activity); it is a no-op for 50Salads/GTEA (their stems encode no view). Every downstream probe
# (order-null AND quality) now runs on the deduped executions.
n_raw = len(X)
meta, X, labels, _dedup_idx = dedup_by_execution(meta, X, labels, NAME)
print(f'leakage guard: {n_raw} camera-view videos -> {len(X)} independent executions '
      f'(dedup_by_execution; {n_raw - len(X)} near-duplicate views removed)')
cfg = DATASETS[NAME]
ns = meta['n_segments'].to_numpy()
display(pd.DataFrame({
    'metric': ['N independent executions (deduped)', 'N raw camera-view videos',
               '|V| = G (incl. background 0)', '#activity classes',
               'sequence length (segments) p10/50/90', 'published N', 'published #classes'],
    'value':  [len(X), n_raw, G, len(set(labels)),
               tuple(np.percentile(ns, [10, 50, 90]).round(1)),
               cfg['n_videos'], cfg['n_classes']],
}))
print('activities:', sorted(set(labels)))
print('per-activity video counts:', dict(Counter(labels)))

,metric,value
0,N videos (loaded),1712
1,|V| = G (incl. background 0),48
2,#activity classes,10
3,segment length p10/50/90,"(4.0, 6.0, 11.0)"
4,published N,1712
5,published #classes,10


activities: ['cereals', 'coffee', 'friedegg', 'juice', 'milk', 'pancake', 'salat', 'sandwich', 'scrambledegg', 'tea']
per-activity video counts: {'cereals': 184, 'coffee': 167, 'friedegg': 173, 'milk': 187, 'salat': 163, 'sandwich': 169, 'tea': 184, 'pancake': 157, 'scrambledegg': 166, 'juice': 162}


## 2. Co-occurrence ground cost

In [3]:
# Data-driven co-occurrence ground cost over the symbol alphabet (symbols that frequently
# abut are closer) -> shared vocab.compute_distance_matrix. Built once; reused by the suite.
D_G = build_action_seg_ground_cost(NAME, X=X, kind='cooccurrence')
assert np.allclose(D_G, D_G.T) and np.allclose(np.diag(D_G), 0.0), 'D_G must be symmetric, zero-diagonal'
if D_G.shape != (G, G):   # loader G=len(mapping) vs ground-cost G=max(observed)+1 (a top id unused)
    print(f'NOTE: ground-cost G={D_G.shape[0]} != mapping G={G}; using {D_G.shape[0]} for shape-consistency')
    G = D_G.shape[0]
print('D_G shape:', D_G.shape, '| symmetric, zero-diagonal OK')

D_G shape: (48, 48) | symmetric, zero-diagonal OK


## 3. Order-null (headline) — top-6 activities, 15 pairs, 200 shuffles

Frequency-preserving order-shuffle null: `delta_auc = auc_intact - mean(auc_shuffled)` with a permutation
band; `order_helps` iff `ci_low > 0`. Reported as it lands — a positive OR null result is informative.

**Bound (probe-measured):** each of the top-6 activities is capped at **80 videos** (480 total). The full
3×5-fold CV × 200 shuffles × 15 pairs × 4 feature/shuffle combos over |V|=48 (2304-dim transition features)
is ~6.8 h of compute — kept in full at the `order_information` defaults (3x5 CV, `n_shuffles=200`).
**These are the library defaults, not SDS2's protocol** (06f ran 2x5 CV, `n_shuffles=100`; RH §15), so
this run is *more* shuffle-resolved than SDS2 rather than matched to it.

**Leakage caveat (this run predates the fix).** Breakfast films each execution from up to 5 cameras whose
RLE'd GT is near-identical (`dedup_by_execution`: 98.8% of cross-view sequences byte-identical after
RLE; 1712 "videos" = only **503** independent executions). `RepeatedStratifiedKFold` is not group-aware,
so the 80-file/class cap below places duplicate views of one execution in train *and* test.

**The 0/60 verdict survives, but by direction-of-bias — not by cancellation.** It is tempting to say
ΔAUC is self-controlling because leakage inflates intact and shuffled alike; **that is wrong**.
`order_shuffle_null` permutes each sequence *independently*, so duplicate views share an exact feature
vector **only in the intact arm** — the shuffle destroys the memorisation channel. Leakage therefore
biases ΔAUC *upward*, toward a false "order helps". (The canonical "the null is its own control"
argument — `order_evaluation.py`'s docstring, RH §15 — covers classifier **optimism**, which is shared
and does cancel; it does not cover group-structured duplicate leakage.) Since leakage can only
manufacture false **positives** and never a null, an all-null **0/60 is conservative** and stands.
What is not trustworthy is the reported `auc_intact` magnitude and N, which are per-file, not
per-execution. The clean fix — `action_segmentation.dedup_by_execution` —
postdates this run; re-run the order-null through it to report per-execution numbers.

In [4]:
# Cap each of the top-6 activities at 80 videos (probe-measured ~6.8 h at full 3x5-fold CV,
# n_shuffles=200 = the order_information library default; SDS2 (06f) ran 100 with 2x5 CV -- NOT matched).
CAP_ORD, rng_o = 80, np.random.default_rng(1)
top6 = [a for a, _ in Counter(labels).most_common(6)]
idx6 = []
for a in top6:
    ia = np.where(labels == a)[0]
    idx6 += list(ia if len(ia) <= CAP_ORD else rng_o.choice(ia, CAP_ORD, replace=False))
idx6 = np.array(sorted(idx6))
X6, labels6 = [X[i] for i in idx6], labels[idx6]
print('top-6 activities:', top6, f'| capped N={len(X6)} (<= {CAP_ORD}/class):',
      {a: int((labels6 == a).sum()) for a in top6})
res_ord = run_generalization_suite(X6, labels6, G, D_G, name=NAME, run_quality=False, n_shuffles=200)
res_ord['order'].to_csv(os.path.join(OUT, 'order_null.csv'), index=False)
n_pos = int(res_ord['order']['order_helps'].sum())
print(f"\norder_helps (ci_low > 0): {n_pos}/{len(res_ord['order'])}   [SDS2 was 0/15]")
display(res_ord['order'][['comparison', 'feature', 'shuffle', 'auc_intact',
                          'delta_auc', 'ci_low', 'ci_high', 'p_perm', 'order_helps']].round(3))

top-6 activities: ['milk', 'cereals', 'tea', 'friedegg', 'sandwich', 'coffee'] | capped N=480 (<= 80/class): {'milk': 80, 'cereals': 80, 'tea': 80, 'friedegg': 80, 'sandwich': 80, 'coffee': 80}



order_helps (ci_low > 0): 0/60   [SDS2 was 0/15]


,comparison,feature,shuffle,auc_intact,delta_auc,ci_low,ci_high,p_perm,order_helps
0,cereals_vs_coffee,transition,token,1.0,0.0,0.0,0.0,1.00,False
1,cereals_vs_friedegg,transition,token,1.0,0.0,0.0,0.0,1.00,False
2,cereals_vs_milk,transition,token,1.0,0.0,0.0,0.0,1.00,False
3,cereals_vs_sandwich,transition,token,1.0,0.0,0.0,0.0,1.00,False
4,cereals_vs_tea,transition,token,1.0,0.0,0.0,0.0,1.00,False
5,coffee_vs_friedegg,transition,token,1.0,0.0,0.0,0.0,1.00,False
6,coffee_vs_milk,transition,token,1.0,0.0,0.0,0.0,0.99,False
7,coffee_vs_sandwich,transition,token,1.0,0.0,0.0,0.0,1.00,False
8,coffee_vs_tea,transition,token,1.0,0.0,0.0,0.0,1.00,False
9,friedegg_vs_milk,transition,token,1.0,0.0,0.0,0.0,1.00,False


## 3b. Why 0/60? The frequency-headroom screen

The 0/60 above is **not** evidence that order fails to help — it is an artifact of the test having
**no room to fire**. `headroom_table` measures the out-of-fold **unigram-histogram AUC** (`hist_auc`) —
the exact frequency channel the order-shuffle holds fixed — on the *same* top-6 pairs the null used, and
bands each: `saturated` (`hist_auc >= 0.95`) means frequency alone already separates the pair, so
`delta_auc` is 0 by arithmetic and the null cannot fire even if order carried signal. Gating on `hist_auc`
introduces no selection bias — it is a function of shuffle-*invariant* quantities (a histogram *is* a
multiset), pinned as executable invariants in `tests/test_headroom.py`. The full screen across every cached
task, plus a *constructed* frequency-controlled contrast where order **does** help (friedegg-vs-pancake,
ΔAUC 0.36): notebook **06n**.

In [5]:
# The screen: out-of-fold unigram-histogram AUC (the frequency channel the shuffle holds fixed) on the
# SAME top-6 pairs as the null above. 'saturated' (hist_auc >= 0.95) => zero headroom => the 0/60 is vacuous.
from smartflat.features.symbolic_barycenter.generalization.headroom import headroom_table
# re-derive the null's exact top-6 capped subset (seed=1, cap=80) so the screen rows align 1:1 with it
_rng_s, _idx_s = np.random.default_rng(1), []
for _a in [a for a, _ in Counter(labels).most_common(6)]:
    _ia = np.where(labels == _a)[0]
    _idx_s += list(_ia if len(_ia) <= 80 else _rng_s.choice(_ia, 80, replace=False))
_idx_s = np.array(sorted(_idx_s))
scr = headroom_table([X[i] for i in _idx_s], labels[_idx_s], G, D_G, name='breakfast_top6')
n_sat = int((scr['headroom_band'] == 'saturated').sum())
print(f"headroom screen: {n_sat}/{len(scr)} top-6 pairs 'saturated' (hist_auc >= 0.95)  |  "
      f"hist_auc min={scr.hist_auc.min():.3f} median={scr.hist_auc.median():.3f} max={scr.hist_auc.max():.3f}")
print(f"vocab_jaccard {scr.vocab_jaccard.min():.3f}-{scr.vocab_jaccard.max():.3f} (near-disjoint action vocabularies)")
print("=> frequency saturates the AUC ceiling; the delta-AUC order-null has ZERO headroom, so the")
print("   0/60 above is VACUOUS, not evidence about order. (Full screen + constructed contrast: notebook 06n.)")
display(scr[['comparison', 'hist_auc', 'vocab_jaccard', 'n_shared', 'n_marker_0', 'n_marker_1',
             'headroom_band']].round(3))

headroom screen: 15/15 top-6 pairs 'saturated' (hist_auc >= 0.95)  |  hist_auc min=1.000 median=1.000 max=1.000
vocab_jaccard 0.000-0.250 (near-disjoint action vocabularies)
=> frequency saturates the AUC ceiling; the delta-AUC order-null has ZERO headroom, so the
   0/60 above is VACUOUS, not evidence about order. (Full screen + constructed contrast: notebook 06n.)


,comparison,hist_auc,vocab_jaccard,n_shared,n_marker_0,n_marker_1,headroom_band
0,cereals_vs_coffee,1.0,0.111,1,3,5,saturated
1,cereals_vs_friedegg,1.0,0.000,0,4,8,saturated
2,cereals_vs_milk,1.0,0.143,1,3,3,saturated
3,cereals_vs_sandwich,1.0,0.000,0,4,8,saturated
4,cereals_vs_tea,1.0,0.000,0,4,5,saturated
5,coffee_vs_friedegg,1.0,0.000,0,6,8,saturated
6,coffee_vs_milk,1.0,0.250,2,4,2,saturated
7,coffee_vs_sandwich,1.0,0.000,0,6,8,saturated
8,coffee_vs_tea,1.0,0.222,2,4,3,saturated
9,friedegg_vs_milk,1.0,0.000,0,8,4,saturated


## 4. Representation quality — all 10 activities, <=50/class subsample

In [5]:
CAP, rng = 50, np.random.default_rng(42)
idx = []
for a in sorted(set(labels)):
    ia = np.where(labels == a)[0]
    idx += sorted(ia if len(ia) <= CAP else rng.choice(ia, CAP, replace=False))
idx = np.array(sorted(idx))
Xq, labelsq = [X[i] for i in idx], labels[idx]
print(f'quality subsample: N={len(Xq)} (<= {CAP}/activity x {len(set(labels))} activities)')
res_q = run_generalization_suite(Xq, labelsq, G, D_G, name=NAME, run_order=False, upsample_to=UPSAMPLE)
res_q['quality'].reset_index().to_csv(os.path.join(OUT, 'quality.csv'), index=False)
display(res_q['quality'].round(3))

quality subsample: N=500 (<= 50/activity x 10 activities)


/home/perochon/anaconda3/envs/smartflat_repro/lib/python3.11/site-packages/ot/bregman/_barycenter.py:250: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


metric,inertia_rtwe,inertia_native,freq_fidelity,struct_preservation,entropy_bits,n_distinct,n_segments,stability_inertia_rtwe,stability_histogram,dataset
method,,,,,,,,,,
dba_dtw,4.521,6.641,0.177,1.752,2.184,5.65,7.8,0.295,0.004,breakfast
edit_median,3.432,7.458,0.126,1.369,2.182,5.00,6.5,0.000,0.000,breakfast
k_medoid,3.442,3.442,0.141,1.408,2.152,4.80,6.2,0.000,0.000,breakfast
majority_voting,3.663,0.366,0.154,1.435,2.124,5.00,6.7,0.000,0.000,breakfast
soft_dtw,4.239,-16.710,0.178,1.603,2.201,5.50,7.6,0.319,0.003,breakfast
wasserstein,NaN,0.164,0.095,NaN,2.321,48.00,NaN,NaN,0.000,breakfast


## 5. Prototypical execution per activity (chronograms)

In [6]:
# Prototypical execution per activity: a deterministic edit-median barycenter of each
# activity's executions, rendered as a chronogram strip (reuses plot_cohort_barycenters).
methods = default_baseline_methods(D_G)
code_to_label = {i: n for n, i in
                 read_mapping(os.path.join(default_root(NAME), 'mapping.txt'), cfg['background']).items()}
CAP_CHRONO, rng = 60, np.random.default_rng(0)
proto = {}
for a in sorted(set(labels)):
    ia = np.where(labels == a)[0]
    if len(ia) > CAP_CHRONO:
        ia = rng.choice(ia, CAP_CHRONO, replace=False)
    Xa = np.vstack([upsample_sequence(X[i], UPSAMPLE) for i in ia]).astype(int)
    proto[a] = np.asarray(methods['edit_median']['build'](Xa, 0)).astype(int)
plot_cohort_barycenters(proto, groups=sorted(proto), code_to_label=code_to_label, mask_background=True,
                        title=f'{NAME}: prototypical execution per activity (edit-median barycenter)',
                        savepath=os.path.join(OUT, 'chronograms.png'))
print('saved', os.path.join(OUT, 'chronograms.png'))

saved /home/perochon/data-gold-final/outputs/symbolic_barycenter/generalization/breakfast/chronograms.png


**Result.** `order_helps` = **0/60**. But this is **not** a replication of SDS2's 0/15, and should not
be read as one: SDS2's null had real headroom (`auc_intact` 0.63–0.84 as printed in RH §15; 0.61–0.82 over all 15
rows of this host's regenerated artifact — either way, nowhere near the 1.0 ceiling), whereas here
`auc_intact = 1.0` on **all** 60 cells, so the top-6 activities are **perfectly separable by symbol
frequency alone** (which actions occur / how often); shuffling order — which preserves each sequence's
symbol multiset — leaves AUC at 1.0, hence `delta_auc = 0`.

**Honest caveat.** Frequency *saturates* here (AUC ceiling), so the ΔAUC null has no headroom to reveal
order signal even if some were present — a harder, frequency-controlled comparison (e.g. sub-activities
that share an action vocabulary) would be needed to probe order sensitively. The valid conclusion is that
order is *not required* for activity discrimination on Breakfast, consistent with SDS2.

**For the paper — narrow claim only.** This run supports *"order is **not required** when activity
vocabularies are disjoint"*. It does **not** support "the frequency-only result generalises externally"
(§7.2) and it does **not** support "order never helps": a saturated task yields no evidence about
ordering in either direction. A genuine external order test needs a **frequency-controlled** task
(e.g. sub-activities sharing an action vocabulary). The in-flight `generalization/headroom.py`
(`headroom_table`, `restrict_to_shared_vocabulary`, `pooled_markov_surrogate`) targets exactly this —
note it is **untracked at the time of writing**, so commit it before relying on this pointer.

**The quality table must NOT feed `tab:baselines`.** It is the Tier-D representation-quality harness,
which PAPER_BRIDGE §1 states is *"additive to, not a replacement for, the Tier-A/B discrimination
baselines and must not be pasted into `tab:baselines`"* — and it has no AUC column and no
proposed-method row. A §6.5 Breakfast column would require running the discrimination baseline suite
(TW-TWE + mode-DBA + baselines) on Breakfast, which this notebook does not do.

Read as-is, the quality table ranks averagers on representation fidelity (edit-median / k-medoid most
faithful sequence averagers; wasserstein most frequency-faithful) — but `inertia_native` is **not
comparable across methods** (each is scored on its own native distance; RH §24). The common yardstick
is `inertia_rtwe`. Chronograms show the prototypical recipe execution per activity.
CSVs: `order_null.csv`, `quality.csv`; figure `chronograms.png`.

**Status:** these results are not yet folded into `RESULTS_HANDOFF_barycenters.md` (it ends at §26); an
RH §27 should record the Phase-2 action-seg family with the frequency-saturation diagnosis above.